In [63]:
# add the root to the path
import sys
import os
import pandas as pd
import json
sys.path.append('/Users/pmassaro/Repos/JobSeeker')
from sqlalchemy import text
from jobseeker.scraper.database.database_manager import DatabaseManager
from jobseeker.llm import ModelNames
from jobseeker.llm.cv_data_extractor import CVLLMExtractor
from jobseeker.llm.job_description_extractor import JobDescriptionLLMExtractor
from jobseeker.llm.requirements_qualification_comparator import RequirementsQualificationsComparator
from jobseeker.llm.cover_letter_builder import CoverLetterBuilder

from jobseeker.llm.utils import extract_text

db = DatabaseManager()
jd_extractor = JobDescriptionLLMExtractor(model_name=ModelNames.GPT4_TURBO,temperature=0)


In [25]:
# create a query selecting all the job postings
query = text("SELECT * FROM job_postings where lower(company) like '%grammarly%'" )
job_postings= db.get_session().execute(query)
job_postings = pd.DataFrame(job_postings.fetchall(), columns=job_postings.keys())
job_postings
job_posting_example = job_postings.iloc[2]
job_posting_example

id                                                                      629
job_id                                                           3863257446
title                                 Applied Scientist, Strategic Research
seniority_level                                            Mid-Senior level
employment_type                                                   Full-time
job_description           Grammarly is excited to offer a\n\nremote-firs...
company                                                           Grammarly
company_url                      https://www.linkedin.com/company/grammarly
industries                                           [Software Development]
job_functions                   [Research, Analyst, Information Technology]
institution_id                                                         None
job_salary_range_max                                                    NaN
job_salary_range_min                                                    NaN
job_poster_p

In [26]:
job_posting_example['job_description_structured'] = jd_extractor.extract_data_from_text(text=job_posting_example['job_description'])
trimmed_job_posting_example= job_posting_example[["title","company","seniority_level","employment_type","job_functions","industries","job_description_structured","skills"]]
trimmed_job_posting_example['company_about'] = """
Grammarly is the world’s leading AI writing assistance company trusted by over 30 million people and 70,000 professional teams every day. From instantly creating a first draft to perfecting every message, Grammarly helps people at 96% of the Fortune 500 get their point across—and get results—without compromising security or privacy. We believe that great writing gets work done. 

Grammarly’s product offerings—Grammarly Business, Grammarly Premium, Grammarly Free, and Grammarly for Education—work where you do, delivering contextually relevant writing support across over 500,000 apps and websites. 

Founded in 2009, Grammarly is No. 7 on the Forbes Cloud 100, one of TIME’s 100 Most Influential Companies, one of Fast Company’s Most Innovative Companies in AI, and one of Inc.’s Best Workplaces. We operate with a remote-first hybrid work model, meaning we primarily work from home and meet for in-person collaboration at our hubs in North America and Europe.
"""

trimmed_job_posting_example_json= trimmed_job_posting_example.to_json()
with open("job_posting_example.json", "w") as f:
    f.write(trimmed_job_posting_example_json)


2024-04-18 13:52:52,337 - JobDescriptionExtractor - INFO - Extracted data from text: 
 Tokens Used: 3185
	Prompt Tokens: 2666
	Completion Tokens: 519
Successful Requests: 1
Total Cost (USD): $0.0
/var/folders/kv/r63tylr57mv752m655mj7p040000gn/T/ipykernel_4532/1030586753.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  job_posting_example['job_description_structured'] = jd_extractor.extract_data_from_text(text=job_posting_example['job_description'])
/var/folders/kv/r63tylr57mv752m655mj7p040000gn/T/ipykernel_4532/1030586753.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  job_posting_example['job_description_struct

In [27]:

cv_text = extract_text("/Users/pmassaro/Repos/JobSeeker/jobseeker/media/1/CV.pdf")
cv_extractor = CVLLMExtractor(model_name=ModelNames.GPT4_TURBO,temperature=0)
cv_data = cv_extractor.extract_data_from_text(cv_text)
cv_string = json.dumps(cv_data)
# Save the extracted data to a JSON file
with open("cv_data.json", "w") as f:
    json.dump(cv_data, f, indent=4)
    
cv_data

/Users/pmassaro/miniforge3/envs/jobseeker/lib/python3.11/site-packages/pydantic/json_schema.py:2099: PydanticJsonSchemaWarning: Default value default=PydanticUndefined description='Full name of the individual.' extra={} is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/pmassaro/miniforge3/envs/jobseeker/lib/python3.11/site-packages/pydantic/json_schema.py:2099: PydanticJsonSchemaWarning: Default value description='Contact phone number.' extra={} is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/pmassaro/miniforge3/envs/jobseeker/lib/python3.11/site-packages/pydantic/json_schema.py:2099: PydanticJsonSchemaWarning: Default value description='Email address.' extra={} is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWa

{'profile': {'name': 'Patricio Nicolas Massaro Rocca',
  'contact_number': 'H+1-202-754-6052',
  'email': 'Bpatomassaro@gmail.com',
  'summary': None,
  'address': None},
 'experiences': [{'position_title': 'Lead Machine Learning engineer',
   'company_name': 'Bairesdev',
   'start_date': '2020',
   'end_date': '2024',
   'accomplishments': ["First hire in the Machine Learning department of the company's growth acceleration service, leading a team of 3 people.",
    'Train and implement prospect segmentation and rating models for over a hundred clients using AWS, Databricks, MLFlow, and SparkML, generating 500M daily predictions and doubling marketing campaign conversion rates.',
    'Develop ML-based early detection algorithms, reducing the platform spam rate by 30%.',
    'Design and implement a natural language processing deep learning model to classify inbound messages, reducing human workload by 37%.',
    'Definition of the data architecture to optimize ML models training, increa

In [28]:
comparator = RequirementsQualificationsComparator(model_name=ModelNames.GPT4_TURBO, cv_data=cv_string,temperature=0.3)

result = comparator.compare(text=trimmed_job_posting_example_json)
        
# convert the comparison to a string
result_str = json.dumps(result['requirements_qualification_comparisons'])

#save the result to a json file
with open("comparison.json", "w") as f:
    f.write(json.dumps(result))

/Users/pmassaro/miniforge3/envs/jobseeker/lib/python3.11/site-packages/pydantic/json_schema.py:2099: PydanticJsonSchemaWarning: Default value default=Ellipsis description='The job description requirement' extra={} is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/pmassaro/miniforge3/envs/jobseeker/lib/python3.11/site-packages/pydantic/json_schema.py:2099: PydanticJsonSchemaWarning: Default value default=Ellipsis description='The cv qualification' extra={} is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/pmassaro/miniforge3/envs/jobseeker/lib/python3.11/site-packages/pydantic/json_schema.py:2099: PydanticJsonSchemaWarning: Default value default=Ellipsis description='The cv qualification, re-written to use the job description requirement language style' extra={} is not JSON serializable; excludin

In [29]:
result['requirements_qualification_comparisons']

[{'job_description_requirement': 'modern machine learning methods related to natural language processing',
  'cv_qualification': 'Design and implement a natural language processing deep learning model to classify inbound messages, reducing human workload by 37%.',
  'cv_qualification_written_in_requirement_language': 'Designed and implemented advanced natural language processing models using deep learning techniques to classify inbound messages, significantly reducing human workload.'},
 {'job_description_requirement': 'Deep Learning',
  'cv_qualification': 'Train deep learning models in Pytorch for super-resolution, achieving a 20% error reduction in land surface temperature estimation with applications in wildfire management.',
  'cv_qualification_written_in_requirement_language': 'Trained deep learning models using Pytorch for super-resolution applications in environmental monitoring, achieving significant error reduction in temperature estimations.'},
 {'job_description_requirement

In [30]:
from jobseeker.llm.cover_letter_builder import CoverLetterBuilder

def escape_latex(string):
    # Mapping of LaTeX special characters that need to be escaped
    latex_special_chars = {
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\^{}',
        '\\': r'\textbackslash{}'
    }

    # Replace each special character with its escaped version
    return ''.join(latex_special_chars.get(c, c) for c in string)

builder = CoverLetterBuilder(model_name=ModelNames.GPT4_TURBO,cv_data=cv_string,temperature=0.1)
letter = builder.build_letter(requirement_qualification_comparison=result_str,job_data=trimmed_job_posting_example_json)
letter_escaped = escape_latex(letter)
letter_escaped = [line for line in letter_escaped.split("\n") if line != ""]
letter_escaped
        

2024-04-18 13:54:35,336 - CoverLetterBuilder - INFO - Extracted data from text: 
 Tokens Used: 3924
	Prompt Tokens: 3567
	Completion Tokens: 357
Successful Requests: 1
Total Cost (USD): $0.0


['I am a Lead Machine Learning Engineer with a robust background in deploying advanced machine learning models, particularly in natural language processing and deep learning. My recent project involved designing and implementing a deep learning model for natural language processing that reduced human workload by 37\\%, aligning closely with the requirements of your Applied Scientist position at Grammarly.',
 'Driven by curiosity and a passion for challenging work, I have successfully applied modern machine learning methods to solve complex problems in natural language processing. At Bairesdev, I spearheaded the development of a model that classifies inbound messages, significantly reducing manual intervention. This initiative not only optimized operational efficiency but also allowed for reallocation of resources to more strategic tasks, demonstrating my capability to enhance productivity through innovation.',
 'My experience with deep learning extends to environmental monitoring, wher

In [31]:
letter

'I am a Lead Machine Learning Engineer with a robust background in deploying advanced machine learning models, particularly in natural language processing and deep learning. My recent project involved designing and implementing a deep learning model for natural language processing that reduced human workload by 37%, aligning closely with the requirements of your Applied Scientist position at Grammarly.\n\nDriven by curiosity and a passion for challenging work, I have successfully applied modern machine learning methods to solve complex problems in natural language processing. At Bairesdev, I spearheaded the development of a model that classifies inbound messages, significantly reducing manual intervention. This initiative not only optimized operational efficiency but also allowed for reallocation of resources to more strategic tasks, demonstrating my capability to enhance productivity through innovation.\n\nMy experience with deep learning extends to environmental monitoring, where I t

In [56]:
import subprocess
import glob
import time




    


with open("/Users/pmassaro/Repos/JobSeeker/jobseeker/templates/cover_letter.tex", "r") as f:
    cover_letter = update_tex_template(template=f.read(), paragraphs=letter_escaped, candidate_name=cv_data['profile']['name'])
    create_pdf_from_latex(cover_letter,path=f"/Users/pmassaro/Repos/JobSeeker/jobseeker/media/1/{job_posting_example['job_id']}")

PDF created successfully: /Users/pmassaro/Repos/JobSeeker/jobseeker/media/1/3863257446.pdf


In [59]:
        os.path.normpath(f"/Users/pmassaro/Repos/JobSeeker/jobseeker/media/1/{job_posting_example['job_id']}")
        # os.path.basename(os.path.normpath(f"/Users/pmassaro/Repos/JobSeeker/jobseeker/media/1/{job_posting_example['job_id']}"))
        
        #or not any(char.isdigit() for char in os.path.basename(os.path.normpath(path))):


'/Users/pmassaro/Repos/JobSeeker/jobseeker/media/1/3863257446'

In [62]:
json.dumps(cv_data)

'{"profile": {"name": "Patricio Nicolas Massaro Rocca", "contact_number": "H+1-202-754-6052", "email": "Bpatomassaro@gmail.com", "summary": null, "address": null}, "experiences": [{"position_title": "Lead Machine Learning engineer", "company_name": "Bairesdev", "start_date": "2020", "end_date": "2024", "accomplishments": ["First hire in the Machine Learning department of the company\'s growth acceleration service, leading a team of 3 people.", "Train and implement prospect segmentation and rating models for over a hundred clients using AWS, Databricks, MLFlow, and SparkML, generating 500M daily predictions and doubling marketing campaign conversion rates.", "Develop ML-based early detection algorithms, reducing the platform spam rate by 30%.", "Design and implement a natural language processing deep learning model to classify inbound messages, reducing human workload by 37%.", "Definition of the data architecture to optimize ML models training, increasing performance metrics by 10%.", 